# Macro panel construction

Builds the country-year macro panel from five raw sources, then derives the four panels the
modelling notebooks use.

**Sources** World Bank WDI (net foreign assets, credit-to-GDP), IMF WEO (GDP growth,
inflation, unemployment, current account, nominal GDP), IMF International Liquidity (FX
reserves), BIS (real effective exchange rate, monthly averaged to annual).

**Outputs**

| Panel | Window | Used by |
|---|---|---|
| `macro_base_1999_2017` | 1999-2017 | macro-only baseline |
| `macro_robust_1995_2017` | 1995-2017 | macro-only robustness |
| `macro_combi_base_2004_2017` | 2004-2017 | merge with micro, baseline |
| `macro_combi_robust_2004_2017` | 2004-2017 | merge with micro, robustness |

Scope is the 25 EU countries in the thesis sample. Cyprus and Malta are excluded.

Merged from the original `06_macro_preclean` and `07_macro_panel`.

## Setup

In [ ]:
import re
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

RAW_XLSX = Path("../data/macro/raw/macro_data_1995_2017_combined.xlsx")
PRECLEAN_DIR = Path("../data/macro/preclean")
CLEAN_DIR = Path("../data/macro/clean")

for d in (PRECLEAN_DIR, CLEAN_DIR, CLEAN_DIR / "missings", CLEAN_DIR / "interp", CLEAN_DIR / "parquet_saves"):
    d.mkdir(parents=True, exist_ok=True) #the subfolders have to be created explicitly, otherwise the first save below fails on a fresh checkout

ID_COLS = ["country_iso3", "year"]
YEAR_RANGE = (1995, 2017)

SHEETS = { #the trailing spaces on the last two sheet names are real, they are part of the sheet name in the workbook
    "WDI_NFA":   "WDI_NFA_annual_1995_2017",
    "WDI_DCTPS": "WDI_DCTPS_annual_1995_2017",
    "IMF_WEO":   "IMF_WEO_macro_annual_1995-2017",
    "IMF_IL_FX": "IMF_IL_FX_1995_2017 ",
    "BIS_REER":  "BIS_REER_monthly_timeseries ",
}

VAR_LABELS = { #readable names for the source codes, used for the missingness figure at the end and as documentation of what each code means
    "NGDP_RPCH":          "GDP growth",
    "NGDPD":              "log(Nominal GDP)",
    "PCPIPCH":            "Inflation",
    "LUR":                "Unemployment",
    "BCA_NGDPD":          "Current account (% GDP)",
    "RXF11FX_REVS":       "log(FX reserves level)",
    "BIS_REER":           "REER level",
    "FS.AST.PRVT.GD.ZS":  "Credit-to-GDP level",
    "FM.AST.NFRG.CN":     "Net foreign assets level",
    "reer_g":             "REER growth",
    "fx_g":               "FX reserves growth",
    "nfa_g":              "NFA growth",
    "credit_g":           "Credit-to-GDP growth",
    "credit_gap":         "Credit gap (HP filter)",
}

In [ ]:
# Excludes Cyprus (CYP) and Malta (MLT) per thesis scope.
# The assertion makes it impossible to accidentally run with the wrong list.
EU25_ISO3 = [
    "AUT", "BEL", "BGR", "CZE", "DEU", "DNK", "EST", "ESP",
    "FIN", "FRA", "GRC", "HRV", "HUN", "IRL", "ITA", "LTU",
    "LUX", "LVA", "NLD", "POL", "PRT", "ROU", "SVK", "SVN", "SWE"
]
assert len(EU25_ISO3) == 25, "EU-25 list must have exactly 25 countries"

# BIS uses ISO2 codes; map only the 25 countries we need
ISO2_TO_ISO3 = {
    "AT": "AUT", "BE": "BEL", "BG": "BGR", "CZ": "CZE", "DE": "DEU",
    "DK": "DNK", "EE": "EST", "ES": "ESP", "FI": "FIN", "FR": "FRA",
    "GR": "GRC", "HR": "HRV", "HU": "HUN", "IE": "IRL", "IT": "ITA",
    "LT": "LTU", "LU": "LUX", "LV": "LVA", "NL": "NLD", "PL": "POL",
    "PT": "PRT", "RO": "ROU", "SK": "SVK", "SI": "SVN", "SE": "SWE"
}
assert len(ISO2_TO_ISO3) == 25, "ISO2 mapping must cover exactly 25 countries"

print("[OK] EU-25 scope defined:", EU25_ISO3)

## Knobs

Everything that a robustness check might want to vary sits here rather than being buried in
the code below.

In [ ]:
# User-set knobs
HP_LAMBDA_DEFAULT = 100     # baseline choice, the lower, the tighter the trend, so the 'gap'(actual minus trend) fluctuates more. 1562 is the standard for quarterly data, but we have annual data, so 100 is a common choice. 
HP_LAMBDA_ALT = 1562        # alternative choice for robustness check, it is used to approximate a much smoother trend that captures long financial cycles of 8-15 years instead of business cycles. By having both, we test whether crisis predictability also depends on cycle definition, as is discussed in literature

# Missingness-based interpolation rule for robust datasets:
# interpolate level variables with missingness > 0 and <= this threshold
INTERP_MAX_MISS = 0.12 #the panel is ~19 years, 12% would mean that 2-3 years are missing. If there's only short gaps, we interpolate. If there's long gaps, we dropp them to avoid fabricating trends. If the threshold would be set lower, we would lose too many countries while too high would fabricate the data

# Minimum observations within country for HP filter to compute a gap
HP_MIN_OBS = 6 #HP filter needs to be able to estimate a meaningful trend and avoid edge distortions. With fewer than 6, trends become unstable and the cycle becomes noise. Since we work with annual data, 6 years is a minimum meaningful macro window. Under 6 years, no structural trend is possible while with > 6 years, it would be minimal but usable

## Helper functions

In [ ]:
# Returns a DataFrame with % missing per variable, sorted descending.
# Used at the end to inspect the preclean output.

def missingness_summary(df: pd.DataFrame) -> pd.DataFrame:
    feat_cols = [c for c in df.columns if c not in ID_COLS]
    miss = df[feat_cols].isna().mean().mul(100).round(2).sort_values(ascending=False)
    return miss.reset_index().rename(columns={"index": "variable", 0: "missing_pct"})

In [ ]:
# Utilities
def ensure_sorted(df: pd.DataFrame) -> pd.DataFrame: #sorts by country_iso3 and year before any operation. Important because groupby operations and pct_change depend on correct row order
    return df.sort_values(ID_COLS).reset_index(drop=True)

def missingness_series(df: pd.DataFrame) -> pd.Series:
    feat_cols = [c for c in df.columns if c not in ID_COLS]
    return df[feat_cols].isna().mean().sort_values(ascending=False) #compute % missing per variable and save it as a CSV, used for transparency

def save_missingness_csv(df: pd.DataFrame, out_csv: str):
    miss = (missingness_series(df) * 100).round(2).reset_index()
    miss.columns = ["variable", "missing_pct"]
    miss.to_csv(out_csv, index=False)

def pct_change_by_country(df: pd.DataFrame, col: str) -> pd.Series:
    return df.groupby("country_iso3")[col].pct_change(fill_method=None) #computes year on year growth within each country, without filling missing values (if there's a gap, the growth is NaN, we don't want to fabricate it by filling)

def interpolate_levels(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame: #linear interpolation within each country, internal gaps only (limit_area="inside"). This fills ONLY gaps between two real observations; leading NaNs (before the first real value) and trailing NaNs (after the last) are left as NaN, so we never extrapolate before/after the observed range
    out = ensure_sorted(df.copy())
    cols = [c for c in cols if c in out.columns]
    if not cols:
        return out

    out[cols] = (
        out.groupby("country_iso3")[cols]
           .transform(lambda x: x.interpolate(method="linear", limit_area="inside"))
           .reset_index(level=0, drop=True)
    )
    return out

def add_credit_gap_hp(
    df: pd.DataFrame,
    credit_col: str = "FS.AST.PRVT.GD.ZS", #takes the credit-to-gdp gap for one country
    lambda_hp: float = 100.0,  #using default / base lambda for annual data, but can be changed in one line when calling the function
    out_col: str = "credit_gap", #the gap is actual - trend, and represents how far credit is above/below its long-run trend
    min_obs: int = 6
) -> pd.DataFrame:
    out = ensure_sorted(df.copy())
    if credit_col not in out.columns:
        out[out_col] = np.nan
        return out

    def _gap_for_country(s: pd.Series) -> pd.Series:
        # s is the credit_col Series for one country (passed by transform, not apply)
        s_num = pd.to_numeric(s, errors="coerce")
        if s_num.notna().sum() < min_obs:
            return pd.Series(np.nan, index=s.index)

        # Trim to the range of real data only — no extrapolation before the first real
        # observation or after the last. This follows Drehmann & Juselius (2014) and
        # Alessi & Detken (2017): the HP filter starts from the first real data point
        # and accepts start-point bias rather than fabricating earlier values.
        first_real = s_num.first_valid_index()
        last_real  = s_num.last_valid_index()
        s_trim = s_num.loc[first_real:last_real]

        # Fill only internal gaps within the real data range (no leading/trailing fill)
        s_filled = s_trim.interpolate(method="linear")

        result = pd.Series(np.nan, index=s.index)

        # --- ONE-SIDED / REAL-TIME HP FILTER ------------------------------------
        # The two-sided hpfilter estimates the trend at year t using the WHOLE
        # series, including years t+1..T — i.e. it looks ahead. For an EWS that is
        # future information a policymaker could not have had at year t, so the
        # two-sided gap leaks future data into a predictor. Instead we re-run the
        # HP filter on an EXPANDING window: for each end year k we filter only the
        # data up to and including k, and keep the trend's LAST value — the one
        # point that used no future information. The gap is then
        # actual(k) - trend_realtime(k). This is the standard real-time credit-gap
        # construction (Drehmann & Juselius 2014; the Basel III countercyclical-
        # buffer gap).
        #
        # Consequence: the first (min_obs - 1) years of each country have too little
        # history to estimate a trend, so their gap stays NaN. That is correct — a
        # real-time analyst would not yet have had a stable trend — not a bug.
        # Endpoint (Hamilton 2018) bias in the last estimate is inherent to the HP
        # filter and remains; it is acknowledged in the Limitations section.
        vals = s_filled.to_numpy()
        idx  = s_filled.index
        for k in range(min_obs - 1, len(s_filled)):
            _, trend = sm.tsa.filters.hpfilter(vals[: k + 1], lamb=lambda_hp)
            result.loc[idx[k]] = vals[k] - trend[-1]   # only the endpoint uses no future data
        return result

    # transform applies _gap_for_country to the credit column Series for each country,
    # then reassembles with the original index. country_iso3 is never touched, so it
    # stays as a column — this avoids the pandas 2.2+ groupby.apply() KeyError bug
    # where the groupby key gets absorbed into the index and disappears as a column.
    out[out_col] = out.groupby("country_iso3")[credit_col].transform(_gap_for_country)
    return out

def compute_derived(df: pd.DataFrame, compute_credit_stuff: bool, hp_lambda: float) -> pd.DataFrame:
    """
    Computes derived variables only when their required input columns exist.
    Ensures no KeyErrors if you remove items from candidates.
    """
    out = ensure_sorted(df.copy())

    # REER growth
    if "BIS_REER" in out.columns:
        out["reer_g"] = pct_change_by_country(out, "BIS_REER")

    # FX growth
    if "RXF11FX_REVS" in out.columns:
        out["fx_g"] = pct_change_by_country(out, "RXF11FX_REVS")

    # NFA growth
    if "FM.AST.NFRG.CN" in out.columns:
        out["nfa_g"] = pct_change_by_country(out, "FM.AST.NFRG.CN")

    # Credit growth + HP gap
    if compute_credit_stuff and ("FS.AST.PRVT.GD.ZS" in out.columns):
        out["credit_g"] = pct_change_by_country(out, "FS.AST.PRVT.GD.ZS")
        out = add_credit_gap_hp(out, credit_col="FS.AST.PRVT.GD.ZS", lambda_hp=hp_lambda, out_col="credit_gap", min_obs=HP_MIN_OBS)

    # Log-transform level variables with large cross-country scale differences.
    # Must come AFTER fx_g is computed from the original RXF11FX_REVS values;
    # log-transforming first would corrupt the growth rate calculation.
    if "NGDPD" in out.columns:
        out["NGDPD"] = np.log(out["NGDPD"])

    if "RXF11FX_REVS" in out.columns:
        out["RXF11FX_REVS"] = np.log(out["RXF11FX_REVS"])

    return out

def zero_missing_vars(df: pd.DataFrame, candidates: list[str]) -> list[str]:
    """Return variables among candidates that exist in df and have 0 missingness in df."""
    existing = [c for c in candidates if c in df.columns]
    if not existing:
        return []
    miss = df[existing].isna().mean()
    return [c for c in existing if miss.loc[c] == 0.0]

## 1. Source readers

One function per source. Each returns the same long format, `country_iso3, year, variable,
value`, so they can be stacked.

In [ ]:
# Source 1 & 2: World Bank WDI sheets (wide -> long)
#
# Both WDI sheets share the same structure, so one function handles both.
# Year columns look like '2017 [YR2017]' — we extract the 4-digit year.
# World Bank uses '..' to represent missing values — we replace with NaN.
# The 'Series Code' column becomes the variable name in the long format
# (e.g. 'FM.AST.NFRG.CN' for NFA, 'FS.AST.PRVT.GD.ZS' for credit-to-GDP).

def wdi_to_long(sheet_name: str) -> pd.DataFrame:
    df = pd.read_excel(RAW_XLSX, sheet_name=sheet_name)

    # Identify year columns by their '[YRxxxx]' suffix
    year_cols = [c for c in df.columns if re.match(r"^\d{4}\s*\[YR\d{4}\]$", str(c).strip())]
    if not year_cols:
        raise ValueError(f"No WDI year columns found in sheet '{sheet_name}'")

    long = df.melt(
        id_vars=["Country Code", "Series Code"],
        value_vars=year_cols,
        var_name="year_raw",
        value_name="value"
    )

    # Extract 4-digit year from '2017 [YR2017]'
    long["year"] = long["year_raw"].str.extract(r"(\d{4})")[0].astype(int)

    # Replace WB missing marker '..' with NaN, then coerce to float
    long["value"] = pd.to_numeric(long["value"].replace("..", np.nan), errors="coerce")

    # Drop blank rows from Excel (empty Country Code or Series Code)
    long = long.dropna(subset=["Country Code", "Series Code"])

    return long.rename(columns={
        "Country Code": "country_iso3",
        "Series Code":  "variable"
    })[["country_iso3", "year", "variable", "value"]]

In [ ]:
# Source 3: IMF WEO (wide -> long)
#
# Each row is a country x indicator combination.
# Year columns are plain integers: 1995, 1996, ..., 2017.
# The 'Estimates Start After' and other metadata columns are ignored.
# 'WEO Subject Code' becomes the variable name (e.g. 'NGDP_RPCH', 'LUR').

def weo_to_long() -> pd.DataFrame:
    df = pd.read_excel(RAW_XLSX, sheet_name=SHEETS["IMF_WEO"])

    year_cols = [c for c in df.columns if re.fullmatch(r"\d{4}", str(c).strip())]
    if not year_cols:
        raise ValueError("No year columns found in IMF WEO sheet")

    long = df.melt(
        id_vars=["ISO", "WEO Subject Code"],
        value_vars=year_cols,
        var_name="year",
        value_name="value"
    )
    long["year"]  = long["year"].astype(int)
    # errors='coerce' handles any stray text values (e.g. 'n/a', '--')
    long["value"] = pd.to_numeric(long["value"], errors="coerce")

    # Drop blank rows from Excel (empty ISO or WEO Subject Code)
    long = long.dropna(subset=["ISO", "WEO Subject Code"])

    return long.rename(columns={
        "ISO":              "country_iso3",
        "WEO Subject Code": "variable"
    })[["country_iso3", "year", "variable", "value"]]

In [ ]:
# Source 4: IMF International Liquidity — FX reserves (wide -> long)
#
# Each row is one country; year columns are plain integers.
# 'INDICATOR.ID' is 'RXF11FX_REVS' for all rows (reserves excl. gold).

def il_fx_to_long() -> pd.DataFrame:
    df = pd.read_excel(RAW_XLSX, sheet_name=SHEETS["IMF_IL_FX"])

    year_cols = [c for c in df.columns if re.fullmatch(r"\d{4}", str(c).strip())]
    if not year_cols:
        raise ValueError("No year columns found in IMF IL FX sheet")

    long = df.melt(
        id_vars=["COUNTRY.ID", "INDICATOR.ID"],
        value_vars=year_cols,
        var_name="year",
        value_name="value"
    )
    long["year"]  = long["year"].astype(int)
    long["value"] = pd.to_numeric(long["value"], errors="coerce")

    # Drop blank rows from Excel (empty COUNTRY.ID or INDICATOR.ID)
    long = long.dropna(subset=["COUNTRY.ID", "INDICATOR.ID"])

    return long.rename(columns={
        "COUNTRY.ID":   "country_iso3",
        "INDICATOR.ID": "variable"
    })[["country_iso3", "year", "variable", "value"]]

In [ ]:
# Source 5: BIS REER (monthly long -> annual mean)
#
# BIS stores monthly observations; we aggregate to annual by taking the mean
# across the 12 months of each year. This gives an annual REER level.
#
# Country codes are in 'AT:Austria' format — we split on ':' and map ISO2 -> ISO3.
# Any country not in ISO2_TO_ISO3 (e.g. non-EU25) maps to NaN and is dropped.
#
# Column name format in this sheet is 'COLUMN_ID:Description' (e.g. 'OBS_VALUE:Value').
def bis_reer_to_yearly() -> pd.DataFrame:
    df = pd.read_excel(RAW_XLSX, sheet_name=SHEETS["BIS_REER"])

    df = df[["REF_AREA:Reference area", "TIME_PERIOD:Period", "OBS_VALUE:Value"]].copy()

    # Extract ISO2 from 'AT:Austria' and map to ISO3
    df["iso2"]        = df["REF_AREA:Reference area"].astype(str).str.split(":").str[0].str.strip()
    df["country_iso3"] = df["iso2"].map(ISO2_TO_ISO3)  # NaN for non-EU25

    # Parse month-end dates and extract year
    df["year"]  = pd.to_datetime(df["TIME_PERIOD:Period"], errors="coerce").dt.year
    df["value"] = pd.to_numeric(df["OBS_VALUE:Value"], errors="coerce")

    # Drop rows outside EU-25 scope or with unparseable dates/values
    df = df.dropna(subset=["country_iso3", "year", "value"])

    # Annual mean across all monthly observations per country-year
    yearly = (
        df.groupby(["country_iso3", "year"], as_index=False)["value"]
          .mean()
    )
    yearly["variable"] = "BIS_REER"

    unmapped_share = df["country_iso3"].isna().mean()
    print(f"[BIS] Unmapped ISO2 share (should be ~0): {unmapped_share:.4f}")

    return yearly[["country_iso3", "year", "variable", "value"]]

## 2. Load and stack all five sources

In [ ]:
# Load and reshape all 5 sources
print("Loading WDI NFA (net foreign assets)...")
wdi_nfa = wdi_to_long(SHEETS["WDI_NFA"])
print(f"  shape: {wdi_nfa.shape} | variable: {wdi_nfa['variable'].unique()}")

print("Loading WDI DCTPS (credit-to-GDP)...")
wdi_dct = wdi_to_long(SHEETS["WDI_DCTPS"])
print(f"  shape: {wdi_dct.shape} | variable: {wdi_dct['variable'].unique()}")

print("Loading IMF WEO (GDP, inflation, unemployment, current account, nominal GDP)...")
weo = weo_to_long()
print(f"  shape: {weo.shape} | variables: {sorted(weo['variable'].unique())}")

print("Loading IMF IL FX (FX reserves)...")
ilfx = il_fx_to_long()
print(f"  shape: {ilfx.shape} | variable: {ilfx['variable'].unique()}")

print("Loading BIS REER (monthly -> annual)...")
bis = bis_reer_to_yearly()
print(f"  shape: {bis.shape} | variable: {bis['variable'].unique()}")

# Stack all sources into one long-format DataFrame
macro_long = pd.concat([wdi_nfa, wdi_dct, weo, ilfx, bis], ignore_index=True)
print(f"\n[CONCAT] Combined long format: {macro_long.shape}")

## 3. Filter to the EU-25 and the 1995-2017 window

In [ ]:
# Filter to EU-25 scope and 1995-2017 window
macro_long = macro_long[
    macro_long["country_iso3"].isin(EU25_ISO3) &
    macro_long["year"].between(*YEAR_RANGE)
].copy()

print(f"[FILTER] After EU-25 + year filter: {macro_long.shape}")
print(f"  Countries: {sorted(macro_long['country_iso3'].unique())}")
print(f"  Variables: {sorted(macro_long['variable'].unique())}")

## 4. Duplicate check before pivoting

In [ ]:
# Check for duplicates before pivoting
# pivot_table with aggfunc='first' silently resolves duplicates,
# so we catch them here first to avoid any hidden data issues.
dups = macro_long.duplicated(subset=["country_iso3", "year", "variable"]).sum()
if dups > 0:
    print(f"WARNING: {dups} duplicate (country, year, variable) rows — investigate before proceeding")
    display(macro_long[macro_long.duplicated(subset=["country_iso3", "year", "variable"], keep=False)].head(20))
else:
    print("[OK] No duplicate (country, year, variable) rows")

## 5. Pivot to one row per country-year

In [ ]:
# Pivot long -> wide panel
# Each row becomes one country-year observation.
# Each variable becomes its own column.
macro_preclean = (
    macro_long
    .pivot_table(index=["country_iso3", "year"], columns="variable", values="value", aggfunc="first")
    .reset_index()
)
macro_preclean.columns.name = None  # remove the 'variable' label from the column axis

print(f"[PIVOT] macro_preclean shape: {macro_preclean.shape}")
print(f"Columns: {list(macro_preclean.columns)}")

## 6. Integrity checks

These assertions are the reason the scope cannot drift without the notebook failing loudly.

In [ ]:
assert macro_preclean["country_iso3"].nunique() == 25, "Expected exactly 25 countries"
assert macro_preclean["year"].min() == 1995 and macro_preclean["year"].max() == 2017, "Year range mismatch"
assert macro_preclean.duplicated(subset=ID_COLS).sum() == 0, "Duplicate country-year rows found"

expected_vars = {"NGDP_RPCH", "NGDPD", "PCPIPCH", "LUR", "BCA_NGDPD",
                 "RXF11FX_REVS", "BIS_REER", "FS.AST.PRVT.GD.ZS", "FM.AST.NFRG.CN"}
missing_vars = expected_vars - set(macro_preclean.columns)
assert not missing_vars, f"Missing expected variables: {missing_vars}"

print("[OK] 25 countries")
print("[OK] Year range 1995-2017")
print("[OK] No duplicate country-year rows")
print("[OK] All 9 expected variables present")
print(f"\nFinal shape: {macro_preclean.shape}")  # expected: (575, 11)
print("\nMissingness overview:")
print(missingness_summary(macro_preclean).to_string(index=False))

## 7. Save the preclean panel

In [ ]:
macro_preclean.to_parquet(PRECLEAN_DIR / "macro_preclean.parquet")
print("[SAVED]", PRECLEAN_DIR / "macro_preclean.parquet")

## 8. The credit gap, and the two HP filters

There are **two different HP filters** in this notebook and they are not interchangeable.

`add_credit_gap_hp` is **one-sided (real-time, recursive)** and is the one the thesis uses.
For each end year it refilters the history up to that year and keeps only the last trend
point, so no observation ever uses information from its own future. This is the standard
real-time credit-gap construction of Drehmann and Juselius (2014) and the Basel III
countercyclical buffer gap.

`add_credit_gap_hp_preclean` is **two-sided** and exists only to build the two lambda
comparison files below, which test whether the choice of lambda changes the gap. It uses the
whole series to estimate the trend at every point, so it looks ahead. Do not use its output
as a predictor.

In [ ]:
import statsmodels.api as sm

HP_MIN_OBS = 6  # same threshold as notebook 07: fewer than 6 obs per country → no gap computed

def add_credit_gap_hp_preclean(
    df: pd.DataFrame,
    lambda_hp: float,
    credit_col: str = "FS.AST.PRVT.GD.ZS",
    out_col: str = "credit_gap",
    min_obs: int = HP_MIN_OBS
) -> pd.DataFrame:
    """
    Adds a credit_gap column to df using the HP filter at the given lambda.
    Only uses the credit_col column per country — country_iso3 is never touched.
    Uses transform (not apply) to avoid the pandas 2.2+ groupby KeyError bug.

    Follows Drehmann & Juselius (2014) and Alessi & Detken (2017): the HP filter
    runs only from the first real data point to the last. Years before the first
    real observation and after the last stay NaN — no backward or forward extrapolation.
    """
    out = df.sort_values(ID_COLS).reset_index(drop=True).copy()

    if credit_col not in out.columns:
        out[out_col] = np.nan
        return out

    def _gap_for_country(s: pd.Series) -> pd.Series:
        s_num = pd.to_numeric(s, errors="coerce")
        if s_num.notna().sum() < min_obs:
            return pd.Series(np.nan, index=s.index)

        # Trim to range of real data — no extrapolation before first or after last real value
        first_real = s_num.first_valid_index()
        last_real  = s_num.last_valid_index()
        s_trim = s_num.loc[first_real:last_real]

        # Fill only internal gaps within the real data range (no leading/trailing fill)
        s_filled = s_trim.interpolate(method="linear")

        _, trend = sm.tsa.filters.hpfilter(s_filled, lamb=lambda_hp)

        # Build result: NaN everywhere, then fill in only the trimmed range
        result = pd.Series(np.nan, index=s.index)
        result.loc[s_filled.index] = (s_filled - trend).values
        return result

    out[out_col] = out.groupby("country_iso3")[credit_col].transform(_gap_for_country)
    return out


# Build and save both lambda variants
for lam, suffix in [(100, "hp100"), (1562, "hp1562")]:
    df_hp = add_credit_gap_hp_preclean(macro_preclean, lambda_hp=lam)
    out_path = f"{PRECLEAN_DIR}/macro_preclean_{suffix}.parquet"
    df_hp.to_parquet(out_path)
    n_filled = df_hp["credit_gap"].notna().sum()
    print(f"[SAVED] {out_path} | shape: {df_hp.shape} | credit_gap non-null: {n_filled}")

## 9. Candidate level variables

In [ ]:
# Candidate level variables
# (edit these safely; everything else adapts automatically)
CORE_LEVEL_CANDIDATES = [
    "NGDP_RPCH",      # real GDP growth (already a rate, but can be treated as series)
    "PCPIPCH",        # inflation
    "LUR",            # unemployment
    "BCA_NGDPD",      # current account balance (% GDP)
    "BIS_REER",       # REER level
    "RXF11FX_REVS",   # FX reserves level
    "NGDPD",          # nominal GDP (optional; stays only if 0-missing in baseline window)
]

OPTIONAL_LEVEL_CANDIDATES = [
    "FS.AST.PRVT.GD.ZS",  # credit-to-GDP level
    "FM.AST.NFRG.CN",     # net foreign assets level
]

In [ ]:
macro_preclean = ensure_sorted(macro_preclean)
print("[LOAD] macro_preclean:", macro_preclean.shape)

## 10. Missingness drives the variable selection

Which variables enter each panel is decided by their missingness in that panel's own window,
not by a fixed list.

In [ ]:
pre_1995_2017 = macro_preclean[macro_preclean["year"].between(1995, 2017)].copy()
pre_1999_2017 = macro_preclean[macro_preclean["year"].between(1999, 2017)].copy()
pre_2004_2017 = macro_preclean[macro_preclean["year"].between(2004, 2017)].copy()

save_missingness_csv(pre_1995_2017, f"{CLEAN_DIR}/missings/macro_missingness_1995_2017_preclean.csv")
save_missingness_csv(pre_1999_2017, f"{CLEAN_DIR}/missings/macro_missingness_1999_2017_preclean.csv")
save_missingness_csv(pre_2004_2017, f"{CLEAN_DIR}/missings/macro_missingness_2004_2017_preclean.csv")

print("[MISS] saved preclean missingness overviews (1995/1999/2004 windows).")

# Determine "0-missing core" in baseline windows (data-driven)
BASE99_ZERO = zero_missing_vars(pre_1999_2017, CORE_LEVEL_CANDIDATES)
BASE04_ZERO = zero_missing_vars(pre_2004_2017, CORE_LEVEL_CANDIDATES)

print("\n[BASE99] 0-missing core levels:", BASE99_ZERO)
print("[BASE04] 0-missing core levels:", BASE04_ZERO)

## 11. Macro-only baseline, 1999-2017

Only the core levels that are complete in this window, plus the derived variables that can be
built from them. No credit gap, since the credit series is not complete here.

In [ ]:
macro_base_1999_2017 = pre_1999_2017[ID_COLS + BASE99_ZERO].copy()
macro_base_1999_2017 = compute_derived(
    macro_base_1999_2017,
    compute_credit_stuff=False,   # baseline core should be 0-missing driven; it only includes derived variables from the core dataset that are not 0
    hp_lambda=HP_LAMBDA_DEFAULT
)

# Save + missingness
macro_base_1999_2017.to_parquet(f"{CLEAN_DIR}/parquet_saves/macro_base_1999_2017.parquet")
macro_base_1999_2017.to_csv(f"{CLEAN_DIR}/macro_base_1999_2017.csv", index=False) #also save as CSV for Stata compatibility
save_missingness_csv(macro_base_1999_2017, f"{CLEAN_DIR}/missings/macro_base_missing_1999_2017.csv")
print("[SAVE] macro_base_1999_2017:", macro_base_1999_2017.shape)

## 12. Macro-only robustness, 1995-2017

Core plus optional levels, interpolated where missingness allows, and the derived variables
computed after interpolation so the growth rates are built on the filled series.

In [ ]:
macro_robust_levels_1995_2017 = pre_1995_2017[ID_COLS + [c for c in (CORE_LEVEL_CANDIDATES + OPTIONAL_LEVEL_CANDIDATES) if c in pre_1995_2017.columns]].copy() #takes the raw preclean data filtered to 1995-2017 and grabs all candidate level variables, both core and optional. 
macro_robust_levels_1995_2017 = ensure_sorted(macro_robust_levels_1995_2017)

# Decide which level cols to interpolate (data-driven)
miss_robust = macro_robust_levels_1995_2017.drop(columns=ID_COLS).isna().mean() #calculates missingness for every variable, only interpolates variables where missingness is between 0 and 12. Variables with 0 missingness are already in the core dataset, so they are not interpolated here. Variables with >12% missingness are dropped to avoid fabricating too much data. The rest is interpolated to create a "robust" dataset that balances coverage and data quality.
interp_cols = [c for c in miss_robust.index if (miss_robust[c] > 0) and (miss_robust[c] <= INTERP_MAX_MISS)] 

# Save interpolation decision for transparency
pd.DataFrame({
    "variable": interp_cols,
    "missing_share": miss_robust.loc[interp_cols].values
}).to_csv(f"{CLEAN_DIR}/interp/macro_interp_cols_1995_2017.csv", index=False)

# Log before/after NA counts
na_before = macro_robust_levels_1995_2017[interp_cols].isna().sum() if interp_cols else pd.Series(dtype=int)

macro_robust_levels_1995_2017_interp = interpolate_levels(macro_robust_levels_1995_2017, interp_cols)

na_after = macro_robust_levels_1995_2017_interp[interp_cols].isna().sum() if interp_cols else pd.Series(dtype=int)
filled_log = pd.DataFrame({
    "variable": interp_cols,
    "na_before": na_before.values if len(na_before) else [],
    "na_after": na_after.values if len(na_after) else [],
    "filled": (na_before - na_after).values if len(na_before) else []
})
filled_log.to_csv(f"{CLEAN_DIR}/interp/macro_interp_filled_counts_1995_2017.csv", index=False)

# Derived vars after interpolation (includes credit_gap if credit level exists)
macro_robust_1995_2017 = compute_derived(
    macro_robust_levels_1995_2017_interp,
    compute_credit_stuff=True, #the robust dataset DOES include the credit-to-GDP level, so it can compute credit_g and credit_gap using the LAMBDA_DEFAULT
    hp_lambda=HP_LAMBDA_DEFAULT
)

# Save + missingness
macro_robust_1995_2017.to_parquet(f"{CLEAN_DIR}/parquet_saves/macro_robust_1995_2017.parquet")
macro_robust_1995_2017.to_csv(f"{CLEAN_DIR}/macro_robust_1995_2017.csv", index=False) #also save as CSV for Stata compatibility
save_missingness_csv(macro_robust_1995_2017, f"{CLEAN_DIR}/missings/macro_robust_missing_1995_2017_after_interp.csv")
print("[SAVE] macro_robust_1995_2017:", macro_robust_1995_2017.shape)

## 13. Baseline panel for merging with micro, 2004-2017

In [ ]:
macro_combi_base_2004_2017 = pre_2004_2017[ID_COLS + BASE04_ZERO].copy()
macro_combi_base_2004_2017 = compute_derived(
    macro_combi_base_2004_2017,
    compute_credit_stuff=False,
    hp_lambda=HP_LAMBDA_DEFAULT
)

macro_combi_base_2004_2017.to_parquet(f"{CLEAN_DIR}/parquet_saves/macro_combi_base_2004_2017.parquet")
macro_combi_base_2004_2017.to_csv(f"{CLEAN_DIR}/macro_combi_base_2004_2017.csv", index=False) #also save as CSV for Stata compatibility
save_missingness_csv(macro_combi_base_2004_2017, f"{CLEAN_DIR}/missings/macro_combi_missing_2004_2017.csv")
print("[SAVE] macro_combi_base_2004_2017:", macro_combi_base_2004_2017.shape)

## 14. Robustness panel for merging with micro, 2004-2017

In [ ]:
macro_robust_levels_2004_2017 = pre_2004_2017[ID_COLS + [c for c in (CORE_LEVEL_CANDIDATES + OPTIONAL_LEVEL_CANDIDATES) if c in pre_2004_2017.columns]].copy()
macro_robust_levels_2004_2017 = ensure_sorted(macro_robust_levels_2004_2017)

miss_combo_robust = macro_robust_levels_2004_2017.drop(columns=ID_COLS).isna().mean()
interp_cols_2004 = [c for c in miss_combo_robust.index if (miss_combo_robust[c] > 0) and (miss_combo_robust[c] <= INTERP_MAX_MISS)]

pd.DataFrame({
    "variable": interp_cols_2004,
    "missing_share": miss_combo_robust.loc[interp_cols_2004].values
}).to_csv(f"{CLEAN_DIR}/interp/macro_interp_cols_2004_2017.csv", index=False)

na_before_2004 = macro_robust_levels_2004_2017[interp_cols_2004].isna().sum() if interp_cols_2004 else pd.Series(dtype=int)
macro_robust_levels_2004_2017_interp = interpolate_levels(macro_robust_levels_2004_2017, interp_cols_2004)
na_after_2004 = macro_robust_levels_2004_2017_interp[interp_cols_2004].isna().sum() if interp_cols_2004 else pd.Series(dtype=int)

filled_log_2004 = pd.DataFrame({
    "variable": interp_cols_2004,
    "na_before": na_before_2004.values if len(na_before_2004) else [],
    "na_after": na_after_2004.values if len(na_after_2004) else [],
    "filled": (na_before_2004 - na_after_2004).values if len(na_before_2004) else []
})
filled_log_2004.to_csv(f"{CLEAN_DIR}/macro_interp_filled_counts_2004_2017.csv", index=False)

macro_combi_robust_2004_2017 = compute_derived(
    macro_robust_levels_2004_2017_interp,
    compute_credit_stuff=True,
    hp_lambda=HP_LAMBDA_DEFAULT
)

# --- Real-time credit gap from the LONGEST available history ------------------
# The one-sided HP filter needs warm-up: with min_obs=6, a gap computed on the
# 2004-2017 slice alone would not start until ~2009, blanking 2004-2008. A
# policymaker in 2004 would have used pre-2004 credit history, so we take the
# credit_gap already estimated on the full 1995-2017 robust panel
# (macro_robust_1995_2017, built earlier in this notebook) and slice it to
# 2004-2017. This gives each country maximum real-time history and fuller
# coverage of the modelling window, while the gap for any single year still uses
# no future information. credit_g (growth) stays as computed on the local window
# since it is purely backward-looking and needs no warm-up.
credit_gap_long = macro_robust_1995_2017[["country_iso3", "year", "credit_gap"]]
macro_combi_robust_2004_2017 = (
    macro_combi_robust_2004_2017
    .drop(columns=["credit_gap"])
    .merge(credit_gap_long, on=["country_iso3", "year"], how="left")
)

macro_combi_robust_2004_2017.to_parquet(f"{CLEAN_DIR}/parquet_saves/macro_combi_robust_2004_2017.parquet")
macro_combi_robust_2004_2017.to_csv(f"{CLEAN_DIR}/macro_combi_robust_2004_2017.csv", index=False) #also save as CSV for Stata compatibility
save_missingness_csv(macro_combi_robust_2004_2017, f"{CLEAN_DIR}/missings/macro_combi_missing_2004_2017.csv")
print("[SAVE] macro_combi_robust_2004_2017:", macro_combi_robust_2004_2017.shape)

## 15. Checks

Read-only. Nothing below this heading changes any panel.

In [ ]:
PANELS = {
    "macro_base_1999_2017":         macro_base_1999_2017,
    "macro_robust_1995_2017":       macro_robust_1995_2017,
    "macro_combi_base_2004_2017":   macro_combi_base_2004_2017,
    "macro_combi_robust_2004_2017": macro_combi_robust_2004_2017,
}

for name, df in PANELS.items():
    print(f"{name:<30} shape={str(df.shape):<10} "
          f"countries={df['country_iso3'].nunique():<3} "
          f"years={df['year'].min()}-{df['year'].max()} "
          f"dup_country_year={df.duplicated(subset=ID_COLS).sum()}")

In [ ]:
for name, df in PANELS.items():
    miss = (df.drop(columns=ID_COLS).isna().mean() * 100).round(2).sort_values(ascending=False)
    print(f"\nMissingness, {name}")
    print(miss.to_string())

### Credit gap against real data

Where does `credit_gap` exist although the underlying credit-to-GDP series is missing? Those
are the country-years where the filter ran on interpolated rather than observed values.

In [ ]:
# Step 1: find every country-year where FS.AST.PRVT.GD.ZS is NaN in the preclean
#         (= no real credit-to-GDP observation exists for that country-year)
credit_missing = macro_preclean[macro_preclean["FS.AST.PRVT.GD.ZS"].isna()][["country_iso3", "year"]].copy()

# Step 2: from the robust 1995 dataset, find where credit_gap IS non-null
credit_gap_present = macro_robust_1995_2017[macro_robust_1995_2017["credit_gap"].notna()][["country_iso3", "year"]].copy()

# Step 3: the overlap = credit_gap was computed despite no real input data
extrapolated = credit_missing.merge(credit_gap_present, on=["country_iso3", "year"], how="inner")

print(f"Total country-years with no real credit-to-GDP data:  {len(credit_missing)}")
print(f"Of those, credit_gap is non-null (extrapolated/interpolated): {len(extrapolated)}")
print(f"\nCountries affected and how many years extrapolated:")
print(extrapolated.groupby("country_iso3")["year"].agg(["count", "min", "max"]).rename(
    columns={"count": "n_years", "min": "first_year", "max": "last_year"}
))

### Missingness overview

In [ ]:
datasets = {
    "Macro-only\n1999-2017\n(baseline)": macro_base_1999_2017,
    "Macro-only\n1995-2017\n(robust)":   macro_robust_1995_2017,
    "Combined\n2004-2017\n(baseline)":   macro_combi_base_2004_2017,
    "Combined\n2004-2017\n(robust)":     macro_combi_robust_2004_2017,
}

THRESHOLD = 9 #the baseline missingness cutoff, shown so it is visible which variables sit the wrong side of it

fig, axes = plt.subplots(1, 4, figsize=(18, 7), sharey=False)
fig.suptitle("Macro panel: missingness per variable per dataset", fontsize=13, fontweight="bold", y=1.01)

for ax, (title, df) in zip(axes, datasets.items()):
    miss = (df.drop(columns=ID_COLS).isna().mean() * 100).round(2).sort_values(ascending=True)
    labels = [VAR_LABELS.get(v, v) for v in miss.index]
    colors = ["#C0392B" if v > THRESHOLD else "#2C3E50" for v in miss.values]

    ax.barh(labels, miss.values, color=colors, edgecolor="white", height=0.6)
    ax.axvline(x=THRESHOLD, color="#E67E22", linestyle="--", linewidth=1.5)
    ax.set_xlim(0, max(35, miss.max() + 5))
    ax.set_xlabel("Missing (%)", fontsize=9)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.tick_params(axis="both", labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

fig.legend(
    handles=[
        mpatches.Patch(color="#2C3E50", label=f"Within {THRESHOLD}% threshold"),
        mpatches.Patch(color="#C0392B", label=f"Above {THRESHOLD}% threshold"),
        plt.Line2D([0], [0], color="#E67E22", linestyle="--", linewidth=1.5, label=f"{THRESHOLD}% threshold line"),
    ],
    loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.04),
)

plt.tight_layout()
plt.savefig(CLEAN_DIR / "missings" / "macro_missingness_overview.png", dpi=150, bbox_inches="tight")
plt.show()